# NB3 · Evaluación: BLEU, ROUGE, BERTScore y benchmarks

Notebook **liviano**. Calculamos las métricas de generación con la librería `evaluate` y vemos el contraste **forma vs. significado**. Cierra con un slice de un benchmark (GLUE).

## 0. Preparación

In [1]:
%pip install -q -U evaluate datasets sacrebleu rouge_score bert_score transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.3 MB/s eta 0:00:00


In [2]:
import evaluate
bleu = evaluate.load('sacrebleu')
rouge = evaluate.load('rouge')
bertscore = evaluate.load('bertscore')
print('métricas cargadas')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


métricas cargadas


## 1. BLEU — precisión de n-gramas (traducción)

Mide el solapamiento de n-gramas con la(s) referencia(s), con *brevity penalty*.

In [3]:
ref = [['El gato negro duerme sobre la alfombra']]   # lista de referencias por ejemplo
pred_buena = ['El gato negro duerme en la alfombra']
pred_mala  = ['Un perro corre en el parque']
print('BLEU (buena):', round(bleu.compute(predictions=pred_buena, references=ref)['score'],1))
print('BLEU (mala) :', round(bleu.compute(predictions=pred_mala,  references=ref)['score'],1))

BLEU (buena): 48.9
BLEU (mala) : 0.0


## 2. ROUGE — cobertura (resumen)

Orientada a *recall*: cuánto del contenido de la referencia aparece en la salida.

In [4]:
ref_r  = ['El transfer learning preentrena una vez y adapta a muchas tareas']
pred_r = ['El transfer learning se preentrena una sola vez y luego se adapta a varias tareas']
print(rouge.compute(predictions=pred_r, references=ref_r))

{'rouge1': np.float64(0.7692307692307692), 'rouge2': np.float64(0.41666666666666663), 'rougeL': np.float64(0.7692307692307692), 'rougeLsum': np.float64(0.7692307692307692)}


## 3. Forma vs. significado — el ejemplo clave

Referencia: **"El auto es rápido"**. Comparamos una **paráfrasis** (mismo sentido, otras palabras) contra una frase **opuesta** (comparte palabras). Miramos qué premia cada métrica.

In [5]:
referencia = 'El auto es rápido'
candidatos = {
    'parafrasis (mismo sentido)': 'El coche es veloz',
    'opuesto (comparte palabras)': 'El auto es lento',
}
import pandas as pd
filas = []
for nombre, c in candidatos.items():
    b  = bleu.compute(predictions=[c], references=[[referencia]])['score']
    r  = rouge.compute(predictions=[c], references=[referencia])['rougeL']
    bs = bertscore.compute(predictions=[c], references=[referencia], lang='es')['f1'][0]
    filas.append({'candidato': nombre, 'BLEU': round(b,1), 'ROUGE-L': round(r,3), 'BERTScore-F1': round(bs,3)})
pd.DataFrame(filas)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,candidato,BLEU,ROUGE-L,BERTScore-F1
0,parafrasis (mismo sentido),19.0,0.444,0.917
1,opuesto (comparte palabras),59.5,0.667,0.932


**Lectura:** BLEU y ROUGE suelen premiar más a la frase *opuesta* (comparte palabras) que a la *paráfrasis* (mismo sentido, otras palabras) — miden **forma**. BERTScore, que compara **embeddings** por similitud coseno, premia la paráfrasis — mide **significado**.

## 4. Benchmark ≠ métrica: un slice de GLUE (SST-2)

Un **benchmark** evalúa la capacidad general sobre un conjunto de tareas; por dentro usa métricas (acá, *accuracy*).

In [6]:
from datasets import load_dataset
from transformers import pipeline

sst2 = load_dataset('nyu-mll/glue', 'sst2', split='validation[:200]')
clf = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')
pred = clf([x['sentence'] for x in sst2])
lab = {'NEGATIVE':0, 'POSITIVE':1}
y_hat = [lab[p['label']] for p in pred]
y = [x['label'] for x in sst2]
acc = evaluate.load('accuracy')
print('Accuracy en SST-2 (200 ej.):', acc.compute(predictions=y_hat, references=y))

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  148kB            

... (7 outputs left out)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Accuracy en SST-2 (200 ej.): {'accuracy': 0.91}


## 5. Ejercicio

Toma las salidas generadas en el NB2 (antes vs. después del fine-tuning) y evalúa cuáles se acercan más a una respuesta de referencia con las tres métricas.

In [7]:
# Tu solución acá
